In [1]:
!ls "/content/drive/MyDrive/Papers/Elder Scam"

 all_conversations.jsonl   dataset_metadata.json       raw_datasets
'COlab Notebook'	   Gemma		       scam_conversations.jsonl
'Data Load'		   legit_conversations.jsonl


In [2]:
# ================================================================
# CELL 1 — Install (LET IT FINISH, do not interrupt!)
# Total time: ~3-5 min on A100
# ================================================================

# Install in correct order — unsloth pulls unsloth-zoo as dep automatically
!pip install -q --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"



# Training stack
!pip install -q --upgrade trl datasets accelerate peft bitsandbytes

# Fix the xformers warning from your log
!pip install -q --no-deps xformers

# Verify
import importlib, sys
for pkg in ["unsloth", "transformers", "trl", "peft", "xformers"]:
    try:
        m = importlib.import_module(pkg)
        print(f"✅ {pkg}: {getattr(m, '__version__', 'unknown')}")
    except Exception as e:
        print(f"❌ {pkg}: {e}")

print("\n⚠  NOW: Runtime → Restart session, then run Cell 2.")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 372.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 343.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 282.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 842.7/842.7 kB 416.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 368.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 342.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 340.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 376.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 396.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 400.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 48.7 

In [3]:
!pip install -q "transformers==5.5.0"

In [ ]:
# ================================================================
# DIAGNOSTIC CELL — Run this INSTEAD of Cell 2
# Copy-paste into a new Colab cell and run it
# Share the FULL output with me
# ================================================================

!pip install -q "transformers==5.5.0"

import unsloth
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
import torch, json, os

print(f"Unsloth {unsloth.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# Check transformers version
import transformers
print(f"Transformers: {transformers.__version__}")

# ---------- Mount + Load data ----------
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

SAVE_DIR = "/content/drive/MyDrive/Papers/Elder Scam"
OUTPUT_DIR = f"{SAVE_DIR}/Gemma"

import pandas as pd
scam_df  = pd.read_json(f"{SAVE_DIR}/scam_conversations.jsonl", lines=True)
legit_df = pd.read_json(f"{SAVE_DIR}/legit_conversations.jsonl", lines=True)
print(f"\nLoaded: {len(scam_df)} scam, {len(legit_df)} legit")
print(f"Scam types: {scam_df['scam_type'].value_counts().to_dict()}")

# Check existing annotations
CKPT_PATH = f"{OUTPUT_DIR}/annotated_data.json"
if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        existing = json.load(f)
    print(f"\nExisting annotations: {len(existing)}")
    for item in existing[:3]:
        print(f"  - is_scam={item['is_scam']}, type={item['scam_type']}, "
              f"integrity={item['annotation'].get('integrity_score', '?')}")
else:
    print("\nNo existing annotations found")

# ---------- Load model ----------
model, tokenizer = FastModel.from_pretrained(
    model_name      = "unsloth/gemma-4-E4B-it",
    max_seq_length  = 4096,
    load_in_4bit    = True,
    dtype           = None,
    full_finetuning = False,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

base_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token
base_tokenizer.padding_side = "left"

print(f"\n✅ Model loaded — GPU mem: {torch.cuda.memory_allocated()/1e9:.1f} GB")
FastModel.for_inference(model)

# ---------- Check tokenizer type ----------
print(f"\n--- TOKENIZER INFO ---")
print(f"tokenizer type: {type(tokenizer)}")
print(f"base_tokenizer type: {type(base_tokenizer)}")
print(f"pad_token: {base_tokenizer.pad_token}")
print(f"eos_token: {base_tokenizer.eos_token}")
print(f"padding_side: {base_tokenizer.padding_side}")

# ---------- Template test ----------
ANNOTATION_SYSTEM_PROMPT = """You are an expert in epistemic security and social engineering analysis.
Analyze phone call transcripts and identify manipulation tactics.

Respond with ONLY valid JSON, no markdown, no backticks.

Output this exact structure:
{
  "vectors": {
    "belief_installation":     {"score": <0-100>, "evidence": "<brief description>"},
    "verification_suppression":{"score": <0-100>, "evidence": "<brief description>"},
    "urgency_fabrication":     {"score": <0-100>, "evidence": "<brief description>"},
    "authority_hijacking":     {"score": <0-100>, "evidence": "<brief description>"},
    "emotional_flooding":      {"score": <0-100>, "evidence": "<brief description>"},
    "exit_path_closure":       {"score": <0-100>, "evidence": "<brief description>"}
  },
  "integrity_score": <0-100>,
  "risk_level": "<CRITICAL|HIGH|MEDIUM|LOW>",
  "explanation": "<2-3 sentence plain-language explanation>",
  "recommended_action": "<specific action to take>"
}
integrity_score: 100=safe, 0=fully compromised."""

def _apply_template(msgs, gen_prompt):
    try:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=gen_prompt, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=gen_prompt)

# ---------- TEST 1: Single scam sample (no batching) ----------
print(f"\n{'='*60}")
print("TEST 1: SINGLE SCAM SAMPLE (no batching)")
print(f"{'='*60}")

test_dialogue = scam_df.iloc[0]["dialogue"][:2000]
print(f"Dialogue length: {len(test_dialogue)} chars")
print(f"First 200 chars: {test_dialogue[:200]}...")

user_prompt = (f"Analyze this SCAM call transcript for epistemic manipulation vectors.\n\n"
               f"TRANSCRIPT:\n{test_dialogue}\n\nRespond with ONLY the JSON object.")
prompt_text = _apply_template([
    {"role": "system", "content": ANNOTATION_SYSTEM_PROMPT},
    {"role": "user",   "content": user_prompt},
], gen_prompt=True)

print(f"\nPrompt tokens (approx): {len(prompt_text.split())}")
print(f"Prompt last 100 chars: ...{prompt_text[-100:]}")

inputs = base_tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=3500).to("cuda")
print(f"Input shape: {inputs.input_ids.shape}")

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=500,
        do_sample=False,
        use_cache=True,
        pad_token_id=base_tokenizer.pad_token_id,
    )

prompt_len = inputs.input_ids.shape[1]
full_resp = base_tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
print(f"\n--- RAW MODEL OUTPUT ({len(full_resp)} chars) ---")
print(full_resp)
print(f"--- END RAW OUTPUT ---")

# Try parsing
def _parse_json(resp):
    if "```json" in resp:
        resp = resp.split("```json")[1].split("```")[0].strip()
    elif "```" in resp:
        parts = resp.split("```")
        if len(parts) >= 2:
            resp = parts[1].strip()
    s, e = resp.find("{"), resp.rfind("}") + 1
    if s >= 0 and e > s:
        try:
            parsed = json.loads(resp[s:e])
            if "vectors" in parsed and "integrity_score" in parsed:
                return parsed
        except json.JSONDecodeError as je:
            print(f"JSON parse error: {je}")
            print(f"Attempted to parse: {resp[s:e][:300]}...")
            return None
    else:
        print(f"No JSON braces found in response")
    return None

result = _parse_json(full_resp)
print(f"\nParse result: {'✅ SUCCESS' if result else '❌ FAILED'}")
if result:
    print(f"  integrity_score: {result.get('integrity_score')}")
    print(f"  risk_level: {result.get('risk_level')}")

# ---------- TEST 2: Single legit sample ----------
print(f"\n{'='*60}")
print("TEST 2: SINGLE LEGIT SAMPLE")
print(f"{'='*60}")

test_legit = legit_df.iloc[0]["dialogue"][:2000]
user_prompt2 = (f"Analyze this LEGITIMATE call transcript for epistemic manipulation vectors.\n\n"
                f"TRANSCRIPT:\n{test_legit}\n\nRespond with ONLY the JSON object.")
prompt_text2 = _apply_template([
    {"role": "system", "content": ANNOTATION_SYSTEM_PROMPT},
    {"role": "user",   "content": user_prompt2},
], gen_prompt=True)

inputs2 = base_tokenizer(prompt_text2, return_tensors="pt", truncation=True, max_length=3500).to("cuda")
with torch.no_grad():
    out2 = model.generate(**inputs2, max_new_tokens=500, do_sample=False,
                          use_cache=True, pad_token_id=base_tokenizer.pad_token_id)

full_resp2 = base_tokenizer.decode(out2[0][inputs2.input_ids.shape[1]:], skip_special_tokens=True).strip()
print(f"\n--- RAW MODEL OUTPUT ({len(full_resp2)} chars) ---")
print(full_resp2)
print(f"--- END RAW OUTPUT ---")

result2 = _parse_json(full_resp2)
print(f"\nParse result: {'✅ SUCCESS' if result2 else '❌ FAILED'}")
if result2:
    print(f"  integrity_score: {result2.get('integrity_score')}")
    print(f"  risk_level: {result2.get('risk_level')}")

# ---------- TEST 3: Batched (size 2) ----------
print(f"\n{'='*60}")
print("TEST 3: BATCHED (size=2)")
print(f"{'='*60}")

dialogues = [scam_df.iloc[0]["dialogue"][:2000], legit_df.iloc[0]["dialogue"][:2000]]
is_scams = [True, False]
prompts = []
for d, s in zip(dialogues, is_scams):
    ctx = "SCAM call" if s else "LEGITIMATE call"
    up = f"Analyze this {ctx} transcript for epistemic manipulation vectors.\n\nTRANSCRIPT:\n{d}\n\nRespond with ONLY the JSON object."
    prompts.append(_apply_template([
        {"role": "system", "content": ANNOTATION_SYSTEM_PROMPT},
        {"role": "user",   "content": up},
    ], gen_prompt=True))

batch_inputs = base_tokenizer(prompts, return_tensors="pt", padding=True,
                              truncation=True, max_length=3500).to("cuda")
print(f"Batch input shape: {batch_inputs.input_ids.shape}")
prompt_len_batch = batch_inputs.input_ids.shape[1]

with torch.no_grad():
    batch_out = model.generate(
        **batch_inputs,
        max_new_tokens=500,
        do_sample=False,
        use_cache=True,
        pad_token_id=base_tokenizer.pad_token_id,
    )

for idx, row in enumerate(batch_out):
    resp = base_tokenizer.decode(row[prompt_len_batch:], skip_special_tokens=True).strip()
    print(f"\n--- BATCH ITEM {idx} RAW OUTPUT ({len(resp)} chars) ---")
    print(resp[:500])
    print(f"--- END ---")
    r = _parse_json(resp)
    print(f"Parse: {'✅ SUCCESS' if r else '❌ FAILED'}")

# ---------- TEST 4: Custom known-good test ----------
print(f"\n{'='*60}")
print("TEST 4: HARDCODED SCAM TEST")
print(f"{'='*60}")

custom = """Caller: Hello, this is Agent Williams from the IRS Criminal Investigation Division.
We have found serious discrepancies in your tax filings from 2023.
A federal arrest warrant has been issued in your name.
You must pay $4,500 immediately via gift cards to resolve this before officers are dispatched.
Do NOT hang up this phone or contact anyone else, as this is a federal matter."""

up_custom = f"Analyze this SCAM call transcript for epistemic manipulation vectors.\n\nTRANSCRIPT:\n{custom}\n\nRespond with ONLY the JSON object."
prompt_custom = _apply_template([
    {"role": "system", "content": ANNOTATION_SYSTEM_PROMPT},
    {"role": "user",   "content": up_custom},
], gen_prompt=True)

inp_custom = base_tokenizer(prompt_custom, return_tensors="pt", truncation=True, max_length=3500).to("cuda")
with torch.no_grad():
    out_custom = model.generate(**inp_custom, max_new_tokens=500, do_sample=False,
                                use_cache=True, pad_token_id=base_tokenizer.pad_token_id)

resp_custom = base_tokenizer.decode(out_custom[0][inp_custom.input_ids.shape[1]:], skip_special_tokens=True).strip()
print(f"\n--- RAW OUTPUT ---")
print(resp_custom)
print(f"--- END ---")
r_custom = _parse_json(resp_custom)
print(f"Parse: {'✅ SUCCESS' if r_custom else '❌ FAILED'}")

# ---------- SUMMARY ----------
print(f"\n{'='*60}")
print("DIAGNOSTIC SUMMARY")
print(f"{'='*60}")
print(f"Transformers:  {transformers.__version__}")
print(f"Unsloth:       {unsloth.__version__}")
print(f"Tokenizer:     {type(base_tokenizer).__name__}")
print(f"Test 1 (single scam):    {'✅' if result else '❌'}")
print(f"Test 2 (single legit):   {'✅' if result2 else '❌'}")
print(f"Test 3 (batch):          see above")
print(f"Test 4 (hardcoded scam): {'✅' if r_custom else '❌'}")
print(f"\n⚠ COPY EVERYTHING ABOVE AND SHARE IT")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth 2026.5.2
GPU: NVIDIA L4
VRAM: 23.7 GB
Transformers: 5.5.0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Loaded: 3562 scam, 2839 legit
Scam types: {'robocall': 845, 'refund': 600, 'support': 600, 'ssn': 600, 'reward': 599, 'unknown': 318}

Existing annotations: 1
  - is_scam=1, type=refund, integrity=35
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://gith

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]


✅ Model loaded — GPU mem: 11.0 GB

--- TOKENIZER INFO ---
tokenizer type: <class 'transformers.models.gemma4.processing_gemma4.Gemma4Processor'>
base_tokenizer type: <class 'transformers.models.gemma.tokenization_gemma.GemmaTokenizer'>
pad_token: <pad>
eos_token: <eos>
padding_side: left

TEST 1: SINGLE SCAM SAMPLE (no batching)
Dialogue length: 2000 chars
First 200 chars: Innocent: Hello.  Suspect: Hi there, my name is Karen Thompson and I'm calling from Financial Services Inc. How are you today, Mr. Johnson? Innocent: Who's giving you my name and how did you get my nu...

Prompt tokens (approx): 460
Prompt last 100 chars: ...sound familiar to you? Innocent: I never pa

Respond with ONLY the JSON object.<turn|>
<|turn>model

Input shape: torch.Size([1, 804])

--- RAW MODEL OUTPUT (1932 chars) ---
{
  "vectors": {
    "belief_installation": {
      "score": 75,
      "evidence": "The caller immediately establishes a plausible scenario (eligible refund) to build trust and belief in the

In [ ]:
# ================================================================
# CELL 2 — FULL PIPELINE (annotate → train → test → export GGUF)
# ALL SAVES TO GOOGLE DRIVE
# Pin transformers FIRST to avoid version mismatch failures
# ================================================================

!pip install -q "transformers==5.5.0"

import unsloth
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

import torch, pandas as pd, json, os, random, gc, time, hashlib, shutil
from tqdm.auto import tqdm

print(f"Unsloth {unsloth.__version__} | GPU: {torch.cuda.get_device_name(0)} "
      f"| VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")

import transformers
print(f"Transformers: {transformers.__version__}")
assert transformers.__version__ == "5.5.0", f"WRONG VERSION: {transformers.__version__} — restart runtime!"

# ---------- Paths (ALL ON DRIVE) ----------
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

SAVE_DIR       = "/content/drive/MyDrive/Papers/Elder Scam"
OUTPUT_DIR     = f"{SAVE_DIR}/Gemma"
CKPT_PATH      = f"{OUTPUT_DIR}/annotated_data.json"
TRAIN_CKPT_DIR = f"{OUTPUT_DIR}/training_checkpoints"
LORA_DIR       = f"{OUTPUT_DIR}/lora_adapter"
GGUF_DIR       = f"{OUTPUT_DIR}/gguf"

for d in [OUTPUT_DIR, TRAIN_CKPT_DIR, LORA_DIR, GGUF_DIR]:
    os.makedirs(d, exist_ok=True)

# ---------- Load + sample ----------
scam_df  = pd.read_json(f"{SAVE_DIR}/scam_conversations.jsonl",  lines=True)
legit_df = pd.read_json(f"{SAVE_DIR}/legit_conversations.jsonl", lines=True)
print(f"Loaded: {len(scam_df)} scam, {len(legit_df)} legit")

SCAM_PER_TYPE = 40
N_LEGIT       = 200

sampled_scam = (scam_df.groupby("scam_type")
                .apply(lambda x: x.sample(n=min(SCAM_PER_TYPE, len(x)), random_state=42),
                       include_groups=False)
                .reset_index(level=0).reset_index(drop=True))
sampled_legit = legit_df.sample(n=min(N_LEGIT, len(legit_df)), random_state=42)
print(f"Sampled: {len(sampled_scam)} scam | {len(sampled_legit)} legit")

# ---------- Load Gemma 4 E4B ----------
model, tokenizer = FastModel.from_pretrained(
    model_name      = "unsloth/gemma-4-E4B-it",
    max_seq_length  = 4096,
    load_in_4bit    = True,
    dtype           = None,
    full_finetuning = False,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

base_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token
base_tokenizer.padding_side = "left"

print(f"✅ Model loaded — GPU mem: {torch.cuda.memory_allocated()/1e9:.1f} GB")

FastModel.for_inference(model)
print("✅ Inference mode enabled")


# ================================================================
# Annotation
# ================================================================
ANNOTATION_SYSTEM_PROMPT = """You are an expert in epistemic security and social engineering analysis.
Analyze phone call transcripts and identify manipulation tactics.

Respond with ONLY valid JSON, no markdown, no backticks.

Output this exact structure:
{
  "vectors": {
    "belief_installation":     {"score": <0-100>, "evidence": "<brief description>"},
    "verification_suppression":{"score": <0-100>, "evidence": "<brief description>"},
    "urgency_fabrication":     {"score": <0-100>, "evidence": "<brief description>"},
    "authority_hijacking":     {"score": <0-100>, "evidence": "<brief description>"},
    "emotional_flooding":      {"score": <0-100>, "evidence": "<brief description>"},
    "exit_path_closure":       {"score": <0-100>, "evidence": "<brief description>"}
  },
  "integrity_score": <0-100>,
  "risk_level": "<CRITICAL|HIGH|MEDIUM|LOW>",
  "explanation": "<2-3 sentence plain-language explanation>",
  "recommended_action": "<specific action to take>"
}
integrity_score: 100=safe, 0=fully compromised."""


def _apply_template(msgs, gen_prompt):
    try:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=gen_prompt, enable_thinking=False)
    except TypeError:
        return tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=gen_prompt)


def _build_prompt(dialogue, is_scam):
    ctx = "SCAM call" if is_scam else "LEGITIMATE call"
    user_prompt = (f"Analyze this {ctx} transcript for epistemic manipulation vectors.\n\n"
                   f"TRANSCRIPT:\n{dialogue[:2000]}\n\nRespond with ONLY the JSON object.")
    return _apply_template([
        {"role": "system", "content": ANNOTATION_SYSTEM_PROMPT},
        {"role": "user",   "content": user_prompt},
    ], gen_prompt=True)


def _parse_json(resp):
    if "```json" in resp:
        resp = resp.split("```json")[1].split("```")[0].strip()
    elif "```" in resp:
        parts = resp.split("```")
        if len(parts) >= 2:
            resp = parts[1].strip()
    s, e = resp.find("{"), resp.rfind("}") + 1
    if s >= 0 and e > s:
        try:
            parsed = json.loads(resp[s:e])
            if "vectors" in parsed and "integrity_score" in parsed:
                return parsed
        except json.JSONDecodeError:
            return None
    return None


def annotate_single(dialogue, is_scam):
    """Single-sample generation — most reliable."""
    prompt = _build_prompt(dialogue, is_scam)
    inputs = base_tokenizer(prompt, return_tensors="pt", truncation=True,
                            max_length=3500).to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens   = 500,
            do_sample        = False,
            use_cache        = True,
            pad_token_id     = base_tokenizer.pad_token_id,
        )
    resp = base_tokenizer.decode(out[0][inputs.input_ids.shape[1]:],
                                 skip_special_tokens=True).strip()
    return _parse_json(resp), resp


def annotate_batch(dialogues, is_scams):
    """Batched generation with per-sample fallback on failure."""
    prompts = [_build_prompt(d, s) for d, s in zip(dialogues, is_scams)]
    inputs = base_tokenizer(prompts, return_tensors="pt", padding=True,
                            truncation=True, max_length=3500).to("cuda")
    prompt_len = inputs.input_ids.shape[1]
    with torch.no_grad():
        outs = model.generate(
            **inputs,
            max_new_tokens   = 500,
            do_sample        = False,
            use_cache        = True,
            pad_token_id     = base_tokenizer.pad_token_id,
        )
    results = []
    raw_responses = []
    for row in outs:
        resp = base_tokenizer.decode(row[prompt_len:], skip_special_tokens=True).strip()
        raw_responses.append(resp)
        results.append(_parse_json(resp))
    return results, raw_responses


# ---- Resume support ----
def _key(t): return hashlib.md5(t.encode()).hexdigest()[:16]

if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        annotated_data = json.load(f)
    done_keys = {_key(d["dialogue"]) for d in annotated_data}
    print(f"Resuming: {len(annotated_data)} already annotated")
else:
    annotated_data, done_keys = [], set()

# Auto-detect GPU and set batch size
GPU_NAME = torch.cuda.get_device_name(0)
if "A100" in GPU_NAME:
    BATCH_SIZE = 4
elif "L4" in GPU_NAME:
    BATCH_SIZE = 2
else:
    BATCH_SIZE = 2
print(f"Using BATCH_SIZE={BATCH_SIZE} for {GPU_NAME}")

fail_log = []  # Track failures for debugging


def annotate_split(df, is_scam, label):
    failed = 0
    rows = [r for _, r in df.iterrows() if _key(r["dialogue"]) not in done_keys]
    if not rows:
        print(f"{label}: all already annotated, skipping")
        return
    pbar = tqdm(range(0, len(rows), BATCH_SIZE), desc=label)
    for i in pbar:
        chunk = rows[i:i+BATCH_SIZE]
        dialogues = [r["dialogue"] for r in chunk]

        try:
            results, raw_responses = annotate_batch(dialogues, [is_scam]*len(chunk))
        except torch.cuda.OutOfMemoryError:
            print(f"\n⚠ OOM on batch at index {i}, falling back to single...")
            torch.cuda.empty_cache()
            results, raw_responses = [], []
            for r in chunk:
                try:
                    res, raw = annotate_single(r["dialogue"], is_scam)
                    results.append(res)
                    raw_responses.append(raw)
                except Exception as e:
                    results.append(None)
                    raw_responses.append(str(e))
                    torch.cuda.empty_cache()

        for j, (r, res) in enumerate(zip(chunk, results)):
            if res:
                annotated_data.append({
                    "dialogue":   r["dialogue"],
                    "is_scam":    int(is_scam),
                    "scam_type":  r["scam_type"] if is_scam else "legitimate",
                    "annotation": res,
                })
                done_keys.add(_key(r["dialogue"]))
            else:
                failed += 1
                # Log first 5 failures for debugging
                if len(fail_log) < 5:
                    raw = raw_responses[j] if j < len(raw_responses) else "N/A"
                    fail_log.append({"label": label, "raw_output_first_300": raw[:300]})

        # Checkpoint every 5 batches
        if (i // BATCH_SIZE) % 5 == 0:
            with open(CKPT_PATH, "w") as f:
                json.dump(annotated_data, f)
        pbar.set_postfix(saved=len(annotated_data), failed=failed)

    print(f"{label} done — saved: {len(annotated_data)}, failed: {failed}")


t0 = time.time()
annotate_split(sampled_scam,  is_scam=True,  label="Scam")
annotate_split(sampled_legit, is_scam=False, label="Legit")

# Final save
with open(CKPT_PATH, "w") as f:
    json.dump(annotated_data, f, indent=2)

n_scam = sum(d['is_scam'] for d in annotated_data)
print(f"\n✅ Annotation done in {(time.time()-t0)/60:.1f} min — "
      f"total {len(annotated_data)} (scam={n_scam}, legit={len(annotated_data)-n_scam})")

if fail_log:
    print(f"\n⚠ {len(fail_log)} failure samples logged:")
    for fl in fail_log:
        print(f"  [{fl['label']}] {fl['raw_output_first_300'][:150]}...")


# ================================================================
# Build training dataset
# ================================================================
TRAINING_SYSTEM_PROMPT = """You are AEGIS (Adaptive Epistemic Guard for Intelligent Scam Defense), an AI that detects epistemic manipulation in phone conversations in real time.

Analyze transcripts for six manipulation vectors:
1. belief_installation — Unverified claims presented as facts
2. verification_suppression — Preventing independent verification
3. urgency_fabrication — Artificial time pressure
4. authority_hijacking — False institutional authority
5. emotional_flooding — Emotional overwhelm to disable reasoning
6. exit_path_closure — Eliminating ability to disengage

Output JSON with vector scores (0-100), evidence, integrity_score (100=safe, 0=compromised), risk_level, explanation, and recommended_action.
Respond ONLY with valid JSON."""

training = []
for item in annotated_data:
    training.append({"conversations": [
        {"role": "system",    "content": TRAINING_SYSTEM_PROMPT},
        {"role": "user",      "content": f"Analyze this phone call for epistemic manipulation:\n\n{item['dialogue'][:2000]}"},
        {"role": "assistant", "content": json.dumps(item["annotation"], indent=2)},
    ]})

random.seed(42)
random.shuffle(training)
split = int(0.9 * len(training))
train_data, eval_data = training[:split], training[split:]
print(f"Train {len(train_data)} | Eval {len(eval_data)}")

# Save splits to Drive
with open(f"{OUTPUT_DIR}/train_data.json", "w") as f:
    json.dump(train_data, f)
with open(f"{OUTPUT_DIR}/eval_data.json", "w") as f:
    json.dump(eval_data, f)
print("✅ Train/eval data saved to Drive")

from datasets import Dataset


def format_for_sft(ex):
    return {"text": [_apply_template(c, gen_prompt=False) for c in ex["conversations"]]}


train_ds = Dataset.from_list(train_data).map(format_for_sft, batched=True,
                                             remove_columns=["conversations"])
eval_ds  = Dataset.from_list(eval_data).map(format_for_sft, batched=True,
                                            remove_columns=["conversations"])

# Free memory before training
del sampled_scam, sampled_legit, scam_df, legit_df, training
gc.collect()
torch.cuda.empty_cache()
print(f"GPU mem after cleanup: {torch.cuda.memory_allocated()/1e9:.1f} GB")


# ================================================================
# LoRA + Train — checkpoints to Drive
# ================================================================
model = FastModel.get_peft_model(
    model,
    r=16, lora_alpha=16, lora_dropout=0, bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
model.print_trainable_parameters()

from trl import SFTTrainer, SFTConfig

# Adjust training batch size for GPU
if "A100" in GPU_NAME:
    TRAIN_BATCH = 2
    GRAD_ACCUM  = 4
elif "L4" in GPU_NAME:
    TRAIN_BATCH = 1
    GRAD_ACCUM  = 8
else:
    TRAIN_BATCH = 1
    GRAD_ACCUM  = 8

print(f"Training config: batch={TRAIN_BATCH}, grad_accum={GRAD_ACCUM} "
      f"(effective batch={TRAIN_BATCH * GRAD_ACCUM})")

trainer = SFTTrainer(
    model         = model,
    tokenizer     = base_tokenizer,
    train_dataset = train_ds,
    eval_dataset  = eval_ds,
    args = SFTConfig(
        dataset_text_field          = "text",
        per_device_train_batch_size = TRAIN_BATCH,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_steps                = 10,
        num_train_epochs            = 3,
        learning_rate               = 2e-4,
        fp16                        = not torch.cuda.is_bf16_supported(),
        bf16                        = torch.cuda.is_bf16_supported(),
        logging_steps               = 5,
        eval_strategy               = "steps",
        eval_steps                  = 50,
        save_strategy               = "steps",
        save_steps                  = 50,
        save_total_limit            = 3,
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        output_dir                  = TRAIN_CKPT_DIR,
        optim                       = "adamw_8bit",
        seed                        = 3407,
        max_seq_length              = 2048,
        report_to                   = "none",
    ),
)

print("🚀 Training — checkpoints saving to Drive...")
stats = trainer.train()
print(f"✅ Training done — steps {stats.global_step} loss {stats.training_loss:.4f}")

# Save training stats
with open(f"{OUTPUT_DIR}/training_stats.json", "w") as f:
    json.dump({
        "steps":       stats.global_step,
        "loss":        stats.training_loss,
        "train_size":  len(train_data),
        "eval_size":   len(eval_data),
        "epochs":      3,
        "lora_r":      16,
        "lr":          2e-4,
        "batch_size":  TRAIN_BATCH,
        "grad_accum":  GRAD_ACCUM,
        "max_seq_len": 2048,
        "gpu":         GPU_NAME,
    }, f, indent=2)
print("✅ Training stats saved to Drive")


# ================================================================
# Sanity test
# ================================================================
FastModel.for_inference(model)


def test_aegis(dialogue, label=""):
    msgs = [
        {"role": "system", "content": TRAINING_SYSTEM_PROMPT},
        {"role": "user",   "content": f"Analyze this phone call for epistemic manipulation:\n\n{dialogue[:1500]}"},
    ]
    text = _apply_template(msgs, gen_prompt=True)
    inp = base_tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=500, do_sample=False,
                             use_cache=True, pad_token_id=base_tokenizer.pad_token_id)
    resp = base_tokenizer.decode(out[0][inp.input_ids.shape[1]:], skip_special_tokens=True).strip()
    print(f"\n{'='*60}\n{label}\n{'='*60}")
    print(resp)
    return resp


custom_scam = """Caller: Hello, this is Agent Williams from the IRS Criminal Investigation Division.
We have found serious discrepancies in your tax filings from 2023.
A federal arrest warrant has been issued in your name.
You must pay $4,500 immediately via gift cards to resolve this before officers are dispatched.
Do NOT hang up this phone or contact anyone else, as this is a federal matter."""

legit_call = """Caller: Hi, this is Dr. Chen's office calling to confirm your appointment
scheduled for Thursday at 2:30 PM. If you need to reschedule, please call us
back at 555-0123 during office hours. Thank you and have a great day!"""

scam_result  = test_aegis(custom_scam, "SCAM TEST")
legit_result = test_aegis(legit_call,  "LEGIT TEST")

with open(f"{OUTPUT_DIR}/sanity_test_results.json", "w") as f:
    json.dump({"scam_test": scam_result, "legit_test": legit_result}, f, indent=2)
print("✅ Sanity test results saved to Drive")


# ================================================================
# Save LoRA adapter to Drive
# ================================================================
model.save_pretrained(LORA_DIR)
base_tokenizer.save_pretrained(LORA_DIR)
print(f"✅ LoRA adapter saved to {LORA_DIR}")


# ================================================================
# Export GGUF to Drive
# ================================================================
print("\nExporting GGUF (q4_k_m) — 5-15 min...")
LOCAL_GGUF = "/content/aegis-gemma4-e4b"

try:
    model.save_pretrained_gguf(LOCAL_GGUF, tokenizer,
                               quantization_method="q4_k_m")
except Exception as e:
    print(f"⚠ Processor export failed ({e}); retrying with base tokenizer...")
    model.save_pretrained_gguf(LOCAL_GGUF, base_tokenizer,
                               quantization_method="q4_k_m")

for f in os.listdir(LOCAL_GGUF):
    src = f"{LOCAL_GGUF}/{f}"
    dst = f"{GGUF_DIR}/{f}"
    print(f"  Copying {f} to Drive...")
    shutil.copy2(src, dst)

gguf_file = next(f for f in os.listdir(GGUF_DIR) if f.endswith(".gguf"))

modelfile = f'''FROM ./{gguf_file}

SYSTEM """{TRAINING_SYSTEM_PROMPT}"""

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
'''
with open(f"{GGUF_DIR}/Modelfile", "w") as f:
    f.write(modelfile)


# ================================================================
# Final summary
# ================================================================
print(f"\n{'='*60}")
print(f"✅ ALL ARTIFACTS SAVED TO GOOGLE DRIVE")
print(f"{'='*60}")
print(f"\n📁 {OUTPUT_DIR}/")

for root, dirs, files in os.walk(OUTPUT_DIR):
    level = root.replace(OUTPUT_DIR, "").count(os.sep)
    indent = "  " * level
    folder = os.path.basename(root)
    if level == 0:
        folder = "Gemma"
    print(f"{indent}📂 {folder}/")
    subindent = "  " * (level + 1)
    for fname in sorted(files):
        fpath = os.path.join(root, fname)
        size = os.path.getsize(fpath) / 1e6
        print(f"{subindent}📄 {fname}  ({size:.1f} MB)")

print(f"\n{'='*60}")
print("To run locally:")
print(f"  1. Download {GGUF_DIR}/ from Drive")
print(f"  2. cd gguf/")
print(f"  3. ollama create aegis -f Modelfile")
print(f"  4. ollama run aegis")
print(f"{'='*60}")

Unsloth 2026.5.2 | GPU: NVIDIA A100-SXM4-80GB | VRAM: 85 GB
Transformers: 5.5.0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded: 3562 scam, 2839 legit
Sampled: 240 scam | 200 legit
==((====))==  Unsloth 2026.5.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/203 [00:00<?, ?B/s]

...

ERROR:unsloth_zoo.llama_cpp:Unsloth: Error during loading or introspecting the original script: Failed to execute module convert_hf_to_gguf_original_gguf_9lutep32 from /root/.unsloth/llama.cpp/original_gguf_9lutep32.py
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/llama_cpp.py", line 894, in _load_module_from_path
    spec.loader.exec_module(module)
  File "<frozen importlib._bootstrap_external>", line 999, in exec_module
  File "<frozen importlib._bootstrap>", line 488, in _call_with_frames_removed
  File "/root/.unsloth/llama.cpp/original_gguf_9lutep32.py", line 18, in <module>
    from conversion import (
ModuleNotFoundError: No module named 'conversion'

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/unsloth_zoo/llama_cpp.py", line 957, in _download_convert_hf_to_gguf_cached
    module = _load_module_from_path(temp_original_file_

⚠ Processor export failed (Unsloth: GGUF conversion failed: Failed during loading/introspection of original script: Failed to execute module convert_hf_to_gguf_original_gguf_9lutep32 from /root/.unsloth/llama.cpp/original_gguf_9lutep32.py); retrying with base tokenizer...
Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 7653.84it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

In [1]:
!pip install -q "transformers==5.5.0"
!pip install -q --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
import os, shutil

from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/Papers/Elder Scam"
LORA_DIR = f"{SAVE_DIR}/Gemma/lora_adapter"
GGUF_DIR = f"{SAVE_DIR}/Gemma/gguf"
os.makedirs(GGUF_DIR, exist_ok=True)

model, tokenizer = FastModel.from_pretrained(
    model_name=LORA_DIR,
    max_seq_length=4096,
    load_in_4bit=False,   # need full precision for GGUF export
    dtype=None,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
base_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

model.save_pretrained_gguf("/content/aegis-gguf", base_tokenizer,
                           quantization_method="q4_k_m")

for f in os.listdir("/content/aegis-gguf"):
    shutil.copy2(f"/content/aegis-gguf/{f}", f"{GGUF_DIR}/{f}")

print("✅ GGUF exported to Drive")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 132.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 400.8 MB/s eta 0:00:00
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
==((====))==  Unsloth 2026.5.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = Fals

model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

...

Unsloth: Copying 1 files from cache to `/content/aegis-gguf`: 100%|██████████| 1/1 [00:35<00:00, 35.42s/it]


Successfully copied all 1 files from cache to `/content/aegis-gguf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:45<00:00, 105.40s/it]


Unsloth: Merge process complete. Saved to `/content/aegis-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes
Unsloth: Successfully installed llama.cpp!
Unsloth: Preparing converter script...


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Gemma4ForConditionalGeneration is not supported for MMPROJ conversion. Converting as text-only model.
Unsloth: Initial conversion completed! Files: ['/content/aegis-gguf_gguf/gemma-4-e4b-it.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['/content/aegis-gguf_gguf/gemma-4-e4b-it.Q4_K_M.gguf']
Unsloth: example usage for text only LLMs: /root/.unsloth/llama.cpp/llama-cli --model /content/aegis-gguf_gguf/gemma-4-e4b-it.Q4_K_M.gguf -p "why is the sky blue?"
Unsloth: Saved Ollama Modelfile to /content/aegis-gguf_gguf/Modelfile
Unsloth: convert model to ollama format by running - ollama create model_name -f /content/aegis-gguf_gguf/Modelfile
✅ GGUF exported to Drive


In [3]:
import os

SAVE_DIR = "/content/drive/MyDrive/Papers/Elder Scam"
GGUF_DIR = f"{SAVE_DIR}/Gemma/gguf"

print("Files in gguf/:")
for f in os.listdir(GGUF_DIR):
    size = os.path.getsize(f"{GGUF_DIR}/{f}") / 1e9
    print(f"  {f}  ({size:.2f} GB)")

# Also check if the GGUF ended up somewhere else
for check_dir in [f"{SAVE_DIR}/Gemma", "/content/drive/MyDrive"]:
    for f in os.listdir(check_dir):
        if f.endswith(".gguf"):
            print(f"\n✅ Found GGUF: {check_dir}/{f}")

Files in gguf/:
  model.safetensors  (15.99 GB)
  generation_config.json  (0.00 GB)
  config.json  (0.00 GB)
  chat_template.jinja  (0.00 GB)
  tokenizer_config.json  (0.00 GB)
  tokenizer.json  (0.03 GB)


In [4]:
# ================================================================
# GGUF EXPORT — Fix for failed conversion
# Run this in Colab with GPU runtime
# ================================================================

!pip install -q "transformers==5.5.0"
!pip install -q --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

from unsloth import FastModel
from unsloth.chat_templates import get_chat_template
import os, shutil

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

SAVE_DIR = "/content/drive/MyDrive/Papers/Elder Scam"
LORA_DIR = f"{SAVE_DIR}/Gemma/lora_adapter"
GGUF_DIR = f"{SAVE_DIR}/Gemma/gguf"

# Load from LoRA adapter (not the broken gguf folder)
model, tokenizer = FastModel.from_pretrained(
    model_name=LORA_DIR,
    max_seq_length=4096,
    load_in_4bit=False,
    dtype=None,
)
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")
base_tokenizer = getattr(tokenizer, "tokenizer", tokenizer)

# Try export — if the default method fails, use fallback
LOCAL_GGUF = "/content/aegis-gguf-export"

try:
    model.save_pretrained_gguf(LOCAL_GGUF, base_tokenizer,
                               quantization_method="q4_k_m")
except Exception as e:
    print(f"\n⚠ Method 1 failed: {e}")
    print("Trying manual llama.cpp conversion...\n")

    # Fallback: save merged model, then convert manually
    MERGED_DIR = "/content/aegis-merged"
    model.save_pretrained_merged(MERGED_DIR, base_tokenizer)

    # Install llama.cpp manually
    !cd /content && git clone https://github.com/ggml-org/llama.cpp.git
    !cd /content/llama.cpp && mkdir -p build && cd build && cmake .. && make -j$(nproc) llama-quantize

    # Convert HF to GGUF
    !pip install -q gguf
    !python /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} \
        --outfile /content/aegis-bf16.gguf --outtype bf16

    # Quantize to Q4_K_M
    !/content/llama.cpp/build/bin/llama-quantize \
        /content/aegis-bf16.gguf \
        /content/aegis-q4km.gguf q4_k_m

    os.makedirs(LOCAL_GGUF, exist_ok=True)
    shutil.move("/content/aegis-q4km.gguf", f"{LOCAL_GGUF}/gemma-4-e4b-it.Q4_K_M.gguf")

# ---------- Verify ----------
print("\n" + "="*60)
gguf_files = [f for f in os.listdir(LOCAL_GGUF) if f.endswith(".gguf")]
if gguf_files:
    for f in gguf_files:
        size = os.path.getsize(f"{LOCAL_GGUF}/{f}") / 1e9
        print(f"✅ {f}  ({size:.2f} GB)")

    # Copy to Drive
    os.makedirs(GGUF_DIR, exist_ok=True)
    for f in gguf_files:
        print(f"Copying {f} to Drive...")
        shutil.copy2(f"{LOCAL_GGUF}/{f}", f"{GGUF_DIR}/{f}")

    # Create Modelfile for Ollama
    gguf_name = gguf_files[0]
    modelfile = f'''FROM ./{gguf_name}

SYSTEM """You are AEGIS (Adaptive Epistemic Guard for Intelligent Scam Defense), an AI that detects epistemic manipulation in phone conversations in real time.

Analyze transcripts for six manipulation vectors:
1. belief_installation — Unverified claims presented as facts
2. verification_suppression — Preventing independent verification
3. urgency_fabrication — Artificial time pressure
4. authority_hijacking — False institutional authority
5. emotional_flooding — Emotional overwhelm to disable reasoning
6. exit_path_closure — Eliminating ability to disengage

Output JSON with vector scores (0-100), evidence, integrity_score (100=safe, 0=compromised), risk_level, explanation, and recommended_action.
Respond ONLY with valid JSON."""

PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_ctx 4096
'''
    with open(f"{GGUF_DIR}/Modelfile", "w") as f:
        f.write(modelfile)
    print("✅ Modelfile created")
    print(f"\n✅ ALL DONE — GGUF saved to {GGUF_DIR}")
else:
    print("❌ No .gguf file found — check errors above")
    print(f"Contents of {LOCAL_GGUF}:")
    for f in os.listdir(LOCAL_GGUF):
        print(f"  {f}")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
==((====))==  Unsloth 2026.5.4: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /content/aegis-gguf-export/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/content/aegis-gguf-export`: 100%|██████████| 1/1 [00:35<00:00, 35.15s/it]


Successfully copied all 1 files from cache to `/content/aegis-gguf-export`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:50<00:00, 110.23s/it]


Unsloth: Merge process complete. Saved to `/content/aegis-gguf-export`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Gemma4ForConditionalGeneration is not supported for MMPROJ conversion. Converting as text-only model.
Unsloth: Initial conversion completed! Files: ['/content/aegis-gguf-export_gguf/gemma-4-e4b-it.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed s

In [5]:
import os, shutil

# The GGUF is here (Unsloth adds _gguf suffix to the dir)
ACTUAL_DIR = "/content/aegis-gguf-export_gguf"
GGUF_DIR = "/content/drive/MyDrive/Papers/Elder Scam/Gemma/gguf"

print("Files in actual output dir:")
for f in os.listdir(ACTUAL_DIR):
    size = os.path.getsize(f"{ACTUAL_DIR}/{f}") / 1e9
    print(f"  {f}  ({size:.2f} GB)")

# Copy GGUF + Modelfile to Drive
for f in os.listdir(ACTUAL_DIR):
    if f.endswith(".gguf") or f == "Modelfile":
        print(f"\nCopying {f} to Drive...")
        shutil.copy2(f"{ACTUAL_DIR}/{f}", f"{GGUF_DIR}/{f}")

# Verify
print("\n" + "="*60)
print("Drive gguf/ now contains:")
for f in os.listdir(GGUF_DIR):
    size = os.path.getsize(f"{GGUF_DIR}/{f}") / 1e9
    print(f"  {f}  ({size:.2f} GB)")
print("✅ Done!")

Files in actual output dir:
  gemma-4-e4b-it.Q4_K_M.gguf  (5.34 GB)
  Modelfile  (0.00 GB)

Copying gemma-4-e4b-it.Q4_K_M.gguf to Drive...

Copying Modelfile to Drive...

Drive gguf/ now contains:
  model.safetensors  (15.99 GB)
  generation_config.json  (0.00 GB)
  config.json  (0.00 GB)
  chat_template.jinja  (0.00 GB)
  tokenizer_config.json  (0.00 GB)
  tokenizer.json  (0.03 GB)
  gemma-4-e4b-it.Q4_K_M.gguf  (5.34 GB)
  Modelfile  (0.00 GB)
✅ Done!


In [6]:
# ================================================================
# ZIP only the essential files — run in Colab
# ================================================================

import os, zipfile

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

SAVE_DIR   = "/content/drive/MyDrive/Papers/Elder Scam"
OUTPUT_DIR = f"{SAVE_DIR}/Gemma"
GGUF_DIR   = f"{OUTPUT_DIR}/gguf"
LORA_DIR   = f"{OUTPUT_DIR}/lora_adapter"
ZIP_NAME   = "/content/resources.zip"

files_to_zip = []

# 1. GGUF + Modelfile only (skip the 16GB safetensors)
for f in ["gemma-4-e4b-it.Q4_K_M.gguf", "Modelfile"]:
    fpath = f"{GGUF_DIR}/{f}"
    if os.path.exists(fpath):
        files_to_zip.append((fpath, f"resources/gguf/{f}"))

# 2. LoRA adapter
if os.path.exists(LORA_DIR):
    for f in os.listdir(LORA_DIR):
        files_to_zip.append((f"{LORA_DIR}/{f}", f"resources/lora_adapter/{f}"))

# 3. Data files
for fname in ["annotated_data.json", "train_data.json", "eval_data.json",
              "training_stats.json", "sanity_test_results.json"]:
    fpath = f"{OUTPUT_DIR}/{fname}"
    if os.path.exists(fpath):
        files_to_zip.append((fpath, f"resources/data/{fname}"))

# 4. Original datasets
for fname in ["scam_conversations.jsonl", "legit_conversations.jsonl"]:
    fpath = f"{SAVE_DIR}/{fname}"
    if os.path.exists(fpath):
        files_to_zip.append((fpath, f"resources/data/{fname}"))

# ---------- Show manifest ----------
total = 0
print(f"{'='*60}")
for src, arc in files_to_zip:
    sz = os.path.getsize(src) / 1e6
    total += sz
    print(f"  {arc}  ({sz:.1f} MB)")
print(f"{'='*60}")
print(f"Total: {total:.0f} MB  ({len(files_to_zip)} files)\n")

# ---------- Zip ----------
print("Zipping... (GGUF is 5.3 GB, this takes a few minutes)")
with zipfile.ZipFile(ZIP_NAME, 'w', zipfile.ZIP_STORED) as zf:  # STORED = no compression, faster for large binaries
    for src, arc in files_to_zip:
        print(f"  Adding: {arc}")
        zf.write(src, arc)

print(f"\n✅ {ZIP_NAME}  ({os.path.getsize(ZIP_NAME)/1e9:.2f} GB)")

# Download
from google.colab import files
print("\n⬇️  Starting download...")
files.download(ZIP_NAME)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  resources/gguf/gemma-4-e4b-it.Q4_K_M.gguf  (5335.3 MB)
  resources/gguf/Modelfile  (0.0 MB)
  resources/lora_adapter/README.md  (0.0 MB)
  resources/lora_adapter/adapter_model.safetensors  (169.7 MB)
  resources/lora_adapter/adapter_config.json  (0.0 MB)
  resources/lora_adapter/chat_template.jinja  (0.0 MB)
  resources/lora_adapter/tokenizer.json  (32.2 MB)
  resources/lora_adapter/tokenizer_config.json  (0.0 MB)
  resources/data/annotated_data.json  (1.5 MB)
  resources/data/train_data.json  (1.6 MB)
  resources/data/eval_data.json  (0.2 MB)
  resources/data/training_stats.json  (0.0 MB)
  resources/data/sanity_test_results.json  (0.0 MB)
  resources/data/scam_conversations.jsonl  (6.7 MB)
  resources/data/legit_conversations.jsonl  (4.1 MB)
Total: 5551 MB  (15 files)

Zipping... (GGUF is 5.3 GB, this takes a few minutes)
  Adding: resources/gguf/gemma-4-

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
import shutil
shutil.copy2("/content/resources.zip", "/content/drive/MyDrive/resources.zip")
print(f"✅ Copied to Drive — {os.path.getsize('/content/drive/MyDrive/resources.zip')/1e9:.2f} GB")

✅ Copied to Drive — 5.55 GB


In [9]:
!ls "/content/drive/MyDrive"

gacre_v3_best.pt		 Papers
gacre_v3_embeddings_cache.pt	 resources.zip
gacre_v4_best.pt		 tell-me-a-story-test_encrypted.jsonl
gacre_v4_cache.pt		 tell-me-a-story-test.jsonl
Gadha				 tell-me-a-story-train_encrypted.jsonl
Model				 tell-me-a-story-train.jsonl
nature_order_sensitive_cache.pt  tell-me-a-story-validation_encrypted.jsonl
order_sensitive_best.pt		 tell-me-a-story-validation.jsonl
